# Resumo Detalhado dos Dados PIB (VAB Agropecuário)

Este notebook consolida a análise do Valor Adicionado Bruto (VAB) da agropecuária por município.

In [3]:
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np

# Configuração de caminhos dinâmica
BASE_DIR = Path.cwd().parent.parent
PIB_BRONZE_DIR = BASE_DIR / "data/bronze/pib_vab_agro"
OUTPUT_DIR = BASE_DIR / "data/resume"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def analyze_pib_qualitative(directory):
    files = list(directory.rglob("*.parquet"))
    if not files:
        print(f"❌ Nenhum arquivo .parquet encontrado em {directory}.")
        return None
    
    print(f"🔍 Analisando {len(files)} arquivos de PIB (VAB Agro)...")
    
    all_dfs = []
    for file in tqdm(files, desc="Lendo Parquets"):
        df = pd.read_parquet(file)
        # 1. Limpeza de Metadados SIDRA
        df = df[df['V'] != 'Valor']
        # Adiciona coluna com o nome do arquivo/ano se necessário para rastreabilidade
        all_dfs.append(df)
    
    df_full = pd.concat(all_dfs, ignore_index=True)
    
    # 2. Conversão de Tipos
    # V (Valor) para numérico
    df_full['V'] = pd.to_numeric(df_full['V'], errors='coerce')
    # D3N (Ano) para int
    if 'D3N' in df_full.columns:
        df_full['D3N'] = pd.to_numeric(df_full['D3N'], errors='coerce').astype('Int64')

    print("\n--- 🔍 ANÁLISE QUALITATIVA DO PIB (BRONZE) ---")
    
    # Validação de Nulos em V por Ano
    print("\n⚠️ Validação de Dados Nulos (V) por Ano:")
    null_report = df_full.groupby('D3N')['V'].apply(lambda x: x.isna().sum()).reset_index()
    null_report.columns = ['Ano', 'Qtd_Nulos']
    null_report['Percentual_Nulo'] = round((null_report['Qtd_Nulos'] / df_full.groupby('D3N')['V'].count().values) * 100, 2)
    display(null_report)

    # 1. Variáveis de Estudo (D2N)
    print("\n📊 Variáveis de Estudo (D2N) no PIB:")
    display(df_full['D2N'].unique().tolist())

    # 2. Resumo por Variável e Ano (apenas com dados válidos)
    print("\n📈 Estatísticas de VAB por Ano (Removendo NaNs):")
    df_valid = df_full.dropna(subset=['V'])
    stats_pib = df_valid.groupby(['D3N', 'D2N'])['V'].agg(['count', 'sum', 'mean', 'min', 'max']).reset_index()
    stats_pib.columns = ['Ano', 'Variavel', 'Qtd_Municipios_Validos', 'Soma_VAB', 'Media_VAB', 'Min_VAB', 'Max_VAB']
    display(stats_pib)
    
    # 3. Top 5 Municípios com maior VAB (considerando o último ano com dados válidos)
    if not df_valid.empty:
        latest_valid_year = df_valid['D3N'].max()
        print(f"\n📍 Top 5 Municípios em {latest_valid_year} (pela variável de Valor):")
        top_mun = df_valid[df_valid['D3N'] == latest_valid_year].sort_values(by='V', ascending=False).head(5)
        display(top_mun[['D1N', 'V']])
    else:
        print("\n⚠️ Sem dados numéricos válidos para exibir Ranking de Municípios.")

    return df_full, stats_pib

df_pib, stats_pib = analyze_pib_qualitative(PIB_BRONZE_DIR)

if stats_pib is not None:
    output_path = OUTPUT_DIR / "resumo_consolidado_pib.parquet"
    stats_pib.to_parquet(output_path, index=False)
    print(f"\n✅ Resumos exportados para: {output_path}")


🔍 Analisando 14 arquivos de PIB (VAB Agro)...


Lendo Parquets: 100%|██████████| 14/14 [00:00<00:00, 233.42it/s]


--- 🔍 ANÁLISE QUALITATIVA DO PIB (BRONZE) ---

⚠️ Validação de Dados Nulos (V) por Ano:


,Ano,Qtd_Nulos,Percentual_Nulo
0,2010,5,0.09
1,2011,5,0.09
2,2012,5,0.09
3,2013,0,0.00
4,2014,0,0.00
5,2015,0,0.00
6,2016,0,0.00
7,2017,0,0.00
8,2018,0,0.00
9,2019,0,0.00



📊 Variáveis de Estudo (D2N) no PIB:


...


📈 Estatísticas de VAB por Ano (Removendo NaNs):


,Ano,Variavel,Qtd_Municipios_Validos,Soma_VAB,Media_VAB,Min_VAB,Max_VAB
0,2010,Valor adicionado bruto a preços correntes da a...,5565,159931982.0,28738.900629,0.0,613895.0
1,2011,Valor adicionado bruto a preços correntes da a...,5565,190024024.0,34146.275651,0.0,996289.0
2,2012,Valor adicionado bruto a preços correntes da a...,5565,200695030.0,36063.796945,0.0,1621108.0
3,2013,Valor adicionado bruto a preços correntes da a...,5570,240289996.0,43140.035189,0.0,1305436.0
4,2014,Valor adicionado bruto a preços correntes da a...,5570,249975004.0,44878.815799,0.0,1684344.0
5,2015,Valor adicionado bruto a preços correntes da a...,5570,258967008.0,46493.179174,-2299.0,1773047.0
6,2016,Valor adicionado bruto a preços correntes da a...,5570,306654993.0,55054.756373,0.0,1402282.0
7,2017,Valor adicionado bruto a preços correntes da a...,5570,302971013.0,54393.359605,0.0,1502251.0
8,2018,Valor adicionado bruto a preços correntes da a...,5570,309610996.0,55585.457092,0.0,2482542.0
9,2019,Valor adicionado bruto a preços correntes da a...,5570,310714028.0,55783.487971,0.0,1575333.0



📍 Top 5 Municípios em 2021 (pela variável de Valor):


,D1N,V
66574,Sapezal - MT,5004239.0
66577,Sorriso - MT,4271404.0
63448,São Desidério - BA,4191028.0
66492,Diamantino - MT,4006059.0
66473,Campo Novo do Parecis - MT,3914050.0



✅ Resumos exportados para: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/pib/data/resume/resumo_consolidado_pib.parquet
